# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriKale1328/flyrank-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [50]:
!pip -q install duckdb huggingface_hub pyarrow pandas

In [51]:
import duckdb

print("DuckDB Installed")

DuckDB Installed


In [52]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN is not None)

True


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my selected lane (Refresh / Content Opportunity Scoring), one row represents one content page for one client on one report date.

The unit of analysis is a single content item with its daily search and engagement performance metrics.

For this assignment, I will use a mid-panel month partition (2026-03) instead of the final month to avoid developing inside the natural test window.

The model will use the information available during this period to identify content pages that may require refresh prioritization.

In [53]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

- impressions_90d
- clicks_90d
- ctr
- engagement metrics
- avg_position
- content metadata features
- availability flags (gsc_data_available, ga4_data_available)

These features represent information available before the refresh prioritization decision and can be used by the model to identify content performance patterns.

### Label / Proxy

Refresh priority score (proxy)

The dataset does not contain a direct "refresh needed" label. Therefore, a proxy target will be created using pre-decision historical performance signals to represent whether content may require attention.

### Context

- client identifier
- content identifier
- month
- report date

These fields help organize, join, and split the data but are not used as predictive features.

### Excluded

- trend_direction
- trend_pct
- is_declining_label
- Any future performance information after the decision point

These fields are excluded because they are derived from the outcome or contain future information, which would cause data leakage.

In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [55]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("HuggingFace login successful")


HuggingFace login successful


In [56]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created")

DuckDB connection created


In [57]:
con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("httpfs loaded")

httpfs loaded


In [58]:
con.execute("""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("HF secret created")

HF secret created


In [59]:
query = """
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
"""

df = con.execute(query).df()

df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [60]:
#Query 1: Count + Date Window
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
"""

result = con.execute(query).df()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [61]:
#Query 2: Grain check
query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

duplicates = con.execute(query).df()

duplicates

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


In [62]:
#Query 3: Missing values / availability check.
query = """
SELECT
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available
ORDER BY rows DESC
"""

availability = con.execute(query).df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,rows
0,True,True,False,False,4690323
1,True,True,True,False,1718348
2,True,False,True,<NA>,1528366
3,True,False,False,<NA>,1490375
4,True,True,True,True,364347
5,True,True,False,True,49619


In [63]:
#Query 4: Month window check
query = """
SELECT
    month,
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY month
"""

month_check = con.execute(query).df()

month_check

,month,rows,min_date,max_date
0,2026-03,9841378,2026-03-01,2026-03-31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

This dataset has some limitations that affect what the model can learn:

1. Unbalanced history across clients:
Different clients have different amounts of historical data available. Some clients have longer tracking history, while others have limited data. Therefore, the model cannot assume equal historical context for every client.

2. GSC-only early rows:
Some rows have only Google Search Console (GSC) data available while GA4 data is unavailable. Missing GA4 data does not mean zero user engagement; it means the data source was not available.

3. Window overlaps:
Some aggregated query features are created from fixed time windows that may overlap with the prediction period. These features must be aligned carefully to avoid using future information and causing data leakage.

4. External factors are not captured:
The dataset cannot explain external reasons behind content performance changes, such as marketing campaigns, algorithm updates, seasonal events, or business decisions.

In [64]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.